### Load & Logs

#### So we are planning on using... to load all the compiled program step by step to log the traces

In [2]:
from pathlib import Path
import json, logging
import numpy as np

In [24]:
from tracr.rasp import rasp
from tracr.compiler import compiling, lib, basis_inference, craft_graph_to_model, craft_model_to_transformer, expr_to_craft_graph, nodes, rasp_to_graph, assemble
from tracr.craft import bases, transformers

In [25]:
project_root = Path.cwd()
if not (project_root / "programs.ipynb").exists():
    project_root = project_root / "notebooks"

print(project_root)

/Users/raghul/RaspFormer/notebooks


In [26]:
step1_path = project_root / "programs.ipynb"
print(step1_path)

step1 = json.loads(step1_path.read_text())
#print(step1)

/Users/raghul/RaspFormer/notebooks/programs.ipynb


### Load the existing RASP program definitions without running the old compile,
#### Logging, validation or artifact writing cells.

In [27]:
skip_cells = {9,
              10,
              11,
              12,
              42,
              43    
            }

In [28]:
for i, cell in enumerate(step1["cells"]):
    if cell['cell_type'] != 'code' or i in skip_cells:
        continue

    source = "".join(cell['source'])
    exec(compile(source, f"{step1_path} [cell {i}]", 'exec'), globals())

    

/Users/raghul/RaspFormer/notebooks


In [29]:
# Include the focused A/B examples from Step 1 in this trace as well.
previous_or_self = rasp.Select(
    rasp.indices,
    rasp.indices,
    lambda key, query: (
        key == query - 1 or (query == 0 and key == 0)
    ),
).named("previous_or_self")

In [30]:
prev_token_class = rasp.Aggregate(
    previous_or_self,
    rasp.tokens,
).named("prev_token_class")

In [31]:
histogram = lib.make_hist().named("histogram")

In [32]:
repeated_class = rasp.Map(
    lambda count: 1 if count > 1 else 0,
    histogram,
).named("repeated_class")

In [33]:
STEP1_PROGRAMS = {
    "A_prev_token_class": prev_token_class,
    "B_histogram_then_repeated_class": repeated_class,
}

TRACE_PROGRAMS = {**PROGRAMS, **STEP1_PROGRAMS}

print(f"Loaded {len(PROGRAMS)} existing programs and added A/B: "
      f"{len(TRACE_PROGRAMS)} programs total")

Loaded 12 existing programs and added A/B: 14 programs total


### Doing Tracr things

In [41]:
tracr_samples = [
    [1, 2, 1],
    [-2, 0, 3, 1],
    [5],
    [0, 0, 0, 0],
    [5, 4, 3, 2, 1, 0],
]

atol, rtol = 1e-4, 1e-4
TRACE_ATOL, TRACE_RTOL = atol, rtol
TRACE_SAMPLES = tracr_samples
trace_model = {}

In [42]:
for name, program in TRACE_PROGRAMS.items():
    traced = rasp_to_graph.extract_rasp_graph(program)
    graph, sources, sink = traced.graph, traced.sources, traced.sink

    basis_inference.infer_bases(graph, sink, VOCAB, MAX_SEQ_LEN)
    output_basis = list(graph.nodes[sink[nodes.ID]][nodes.OUTPUT_BASIS])

    expr_to_craft_graph.add_craft_components_to_rasp_graph(
        graph,
        bos_dir=bases.BasisDirection(rasp.tokens.label, BOS),
        mlp_exactness=MLP_EXACTNESS,
    )

    craft_model = craft_graph_to_model.craft_graph_to_model(graph, sources)

    assembled_model = (
        craft_model_to_transformer.craft_model_to_transformer(
            craft_model=craft_model,
            graph=graph,
            sink=sink,
            max_seq_len=MAX_SEQ_LEN,
            compiler_bos=BOS,
            compiler_pad=PAD,
            causal=CASUAL,
        )
    )

    model_config, module_names = (
        assemble._get_model_config_and_module_names(craft_model)
    )
    assert len(module_names) == len(craft_model.blocks)

    token_values = (
        set(graph.nodes[rasp.tokens.label][nodes.VALUE_SET]) | {BOS, PAD}
    )
    tokens_space = bases.VectorSpaceWithBasis.from_values(
        rasp.tokens.label,
        token_values,
    )
    indices_space = bases.VectorSpaceWithBasis.from_values(
        rasp.indices.label,
        range(MAX_SEQ_LEN),
    )
    output_space = bases.VectorSpaceWithBasis(output_basis)

    # This is the same residual space used by Tracr when assembling the model.
    full_space = bases.join_vector_spaces(
        craft_model.residual_space,
        tokens_space,
        indices_space,
        output_space,
    )

    assert [str(direction) for direction in full_space.basis] == (
        assembled_model.residual_labels
    )

    trace_model[name] = {
        "program": program,
        "craft": craft_model,
        "assembled": assembled_model,
        "full_space": full_space,
        "output_space": output_space,
        "module_names": module_names,
    }

In [43]:
print(len(trace_model))
### So its 1 how we got it len of trace_models divs by progras 

14


In [ ]:
def manual_craft_forward(
    craft_model,
    full_space,
    output_space,
    module_names,
    tokens,
    *,
    key_size,
    causal=False,
):
    sequence = [BOS, *tokens]
    x = np.zeros((len(sequence), full_space.num_dims), dtype=np.float64)
    direction_index = full_space.index_by_direction

    for position, token in enumerate(sequence):
        token_direction = bases.BasisDirection(rasp.tokens.label, token)
        x[position, direction_index[token_direction]] = 1.0

        if position > 0:
            index_direction = bases.BasisDirection(
                rasp.indices.label,
                position - 1,
            )
            x[position, direction_index[index_direction]] = 1.0

    one_direction = bases.BasisDirection("one")
    if one_direction in full_space:
        x[:, direction_index[one_direction]] = 1.0

    residual = full_space.make_vector(x)
    block_trace = []

    for module_name, block in zip(module_names, craft_model.blocks):
        block_input = residual.project(block.residual_space)

        if isinstance(
            block,
            (transformers.AttentionHead, transformers.MultiAttentionHead),
        ):
            heads = (
                [block]
                if isinstance(block, transformers.AttentionHead)
                else list(block.heads())
            )

            head_outputs = []

            for head in heads:
                queries = block_input.project(
                    head.w_qk.left_space
                ).magnitudes
                keys = block_input.project(
                    head.w_qk.right_space
                ).magnitudes

                # Match the assembled Transformer's attention scaling.
                logits = (
                    queries @ head.w_qk.matrix @ keys.T
                ) / np.sqrt(key_size)

                if causal:
                    mask = np.tril(
                        np.ones(logits.shape, dtype=bool)
                    )
                    logits = np.where(mask, logits, -1e30)

                # Stable softmax.
                weights = np.exp(
                    logits - logits.max(axis=-1, keepdims=True)
                )
                weights /= weights.sum(axis=-1, keepdims=True)

                values = head.w_ov_residual(block_input).magnitudes
                head_outputs.append(
                    head.residual_space.make_vector(weights @ values)
                )

            block_delta = bases.VectorInBasis.sum(head_outputs)

        elif isinstance(block, transformers.MLP):
            block_delta = block.apply(block_input)

        else:
            raise TypeError(f"Unsupported block: {type(block).__name__}")

        residual = residual + block_delta.project(full_space)

        block_trace.append({
            "module": module_name,
            "residual": residual.magnitudes.copy(),
            "output_scores": residual.project(output_space).magnitudes.copy(),
        })

    return residual, x, block_trace

In [50]:
def values_match(expected, actual):
    if isinstance(expected, (int, float, np.number)) and isinstance(
        actual, (int, float, np.number)
    ):
        return np.isclose(
            expected,
            actual,
            rtol=TRACE_RTOL,
            atol=TRACE_ATOL,
        )
    return expected == actual


evaluator = rasp.DefaultRASPEvaluator()
TRACE_RESULTS = {}
TRACE_SUMMARY = []
TRACE_LOG_ROWS = []
TRACE_RESIDUAL_WARNINGS = []

for name, item in trace_model.items():
    program = item["program"]
    craft_model = item["craft"]
    assembled_model = item["assembled"]
    full_space = item["full_space"]
    output_space = item["output_space"]
    module_names = item["module_names"]

    output_columns = [
        assembled_model.residual_labels.index(str(direction))
        for direction in output_space.basis
    ]

    program_max_error = 0.0
    program_residual_max_error = 0.0
    program_residual_mismatches = 0

    for tokens in TRACE_SAMPLES:
        manual_final, manual_input, block_trace = manual_craft_forward(
    craft_model,
    full_space,
    output_space,
    module_names,
    tokens,
    key_size=assembled_model.model_config.key_size,
    causal=assembled_model.model_config.causal,
)

        actual = assembled_model.apply([BOS, *tokens])

        # Check that the manually created token and position embeddings agree
        # with the assembled model's input embeddings.
        np.testing.assert_allclose(
            manual_input,
            np.asarray(actual.input_embeddings)[0],
            rtol=TRACE_RTOL,
            atol=TRACE_ATOL,
            err_msg=f"{name}, input embeddings, input={tokens}",
        )

        # Compare each manually traced active block with the matching
        # attention/MLP residual recorded by the assembled Transformer.
        # Soft check: log mismatches and continue so final-output checks
        # (scores + RASP decode) still run strictly.
        for step in block_trace:
            _, layer_name, component_name = step["module"].split("/")
            layer_index = int(layer_name.split("_")[-1])
            half_layer_index = 2 * layer_index + (
                1 if component_name == "mlp" else 0
            )

            actual_residual = np.asarray(
                actual.residuals[half_layer_index]
            )[0]

            step_max_error = float(np.max(np.abs(step["residual"] - actual_residual)))
            program_residual_max_error = max(program_residual_max_error, step_max_error)
            if not np.allclose(step["residual"], actual_residual, rtol=TRACE_RTOL, atol=TRACE_ATOL):
                program_residual_mismatches += 1
                TRACE_RESIDUAL_WARNINGS.append({
                    "program": name,
                    "module": step["module"],
                    "tokens": list(tokens),
                    "max_abs_residual_error": step_max_error,
                })
                print(f"WARN {name}, {step['module']} residual mismatch, input={tokens}: max abs diff={step_max_error:.3g}")

        # Tracr disables layer norm in its assembled models, so the final
        # hidden state should equal the manually traced final residual.
        # Soft check (same policy as per-block residuals): log and continue.
        actual_final_residual = np.asarray(actual.transformer_output)[0]
        final_residual_error = float(np.max(np.abs(manual_final.magnitudes - actual_final_residual)))
        program_residual_max_error = max(program_residual_max_error, final_residual_error)
        if not np.allclose(manual_final.magnitudes, actual_final_residual, rtol=TRACE_RTOL, atol=TRACE_ATOL):
            program_residual_mismatches += 1
            TRACE_RESIDUAL_WARNINGS.append({
                "program": name,
                "module": "final_residual",
                "tokens": list(tokens),
                "max_abs_residual_error": final_residual_error,
            })
            print(f"WARN {name}, final_residual mismatch, input={tokens}: max abs diff={final_residual_error:.3g}")

        # These are the pre-argmax output-basis scores (categorical logits,
        # or the output channel for numerical programs).
        manual_scores = manual_final.project(output_space).magnitudes
        model_scores = actual_final_residual[:, output_columns]

        np.testing.assert_allclose(
            manual_scores,
            model_scores,
            rtol=TRACE_RTOL,
            atol=TRACE_ATOL,
            err_msg=f"{name}, output scores, input={tokens}",
        )

        max_error = float(np.max(np.abs(manual_scores - model_scores)))
        program_max_error = max(program_max_error, max_error)

        # Also confirm decoded model outputs still match the RASP evaluator.
        expected = list(evaluator.evaluate(program, tokens))
        decoded = actual.decoded[1:]

        if len(expected) != len(decoded) or not all(
            values_match(left, right)
            for left, right in zip(expected, decoded)
        ):
            raise AssertionError(
                f"{name}, input={tokens}: RASP={expected}, model={decoded}"
            )

        result = {
            "program": name,
            "tokens": list(tokens),
            "max_abs_score_error": max_error,
            "rasp_output": expected,
            "model_decoded": decoded,
            "manual_scores": manual_scores.tolist(),
            "model_scores": model_scores.tolist(),
            "block_trace": [
                {
                    "module": step["module"],
                    "residual": step["residual"].tolist(),
                    "output_scores": step["output_scores"].tolist(),
                }
                for step in block_trace
            ],
        }

        TRACE_RESULTS[(name, tuple(tokens))] = result
        TRACE_LOG_ROWS.append(result)

    TRACE_SUMMARY.append({
        "program": name,
        "inputs_checked": len(TRACE_SAMPLES),
        "max_abs_score_error": program_max_error,
        "residual_max_abs_error": program_residual_max_error,
        "residual_mismatches": program_residual_mismatches,
        "status": "PASS" if program_residual_mismatches == 0 else "PASS_WITH_RESIDUAL_WARN",
    })

    print(
        f"PASS {name}: {len(TRACE_SAMPLES)} inputs; "
        f"max manual/model score error={program_max_error:.3g}; "
        f"residual mismatches={program_residual_mismatches} (max residual diff={program_residual_max_error:.3g})"
    )

display(TRACE_SUMMARY)


PASS P01_absolute: 5 inputs; max manual/model score error=0; residual mismatches=0 (max residual diff=0)
PASS P02_index_parity: 5 inputs; max manual/model score error=0; residual mismatches=0 (max residual diff=0)
PASS P03_increment_by_index: 5 inputs; max manual/model score error=0; residual mismatches=0 (max residual diff=0)
PASS P04_first_element: 5 inputs; max manual/model score error=2.1e-08; residual mismatches=0 (max residual diff=2.1e-08)
PASS P05_histogram: 5 inputs; max manual/model score error=1.91e-06; residual mismatches=0 (max residual diff=1.91e-06)
PASS P06_count_greater_than: 5 inputs; max manual/model score error=1.91e-06; residual mismatches=0 (max residual diff=1.91e-06)
PASS P07_sum_with_next: 5 inputs; max manual/model score error=1.67e-08; residual mismatches=0 (max residual diff=1.91e-06)
PASS P08_pairwise_sum: 5 inputs; max manual/model score error=1.67e-08; residual mismatches=0 (max residual diff=1.91e-06)
PASS P09_check_increasing: 5 inputs; max manual/model

[{'program': 'P01_absolute',
  'inputs_checked': 5,
  'max_abs_score_error': 0.0,
  'residual_max_abs_error': 0.0,
  'residual_mismatches': 0,
  'status': 'PASS'},
 {'program': 'P02_index_parity',
  'inputs_checked': 5,
  'max_abs_score_error': 0.0,
  'residual_max_abs_error': 0.0,
  'residual_mismatches': 0,
  'status': 'PASS'},
 {'program': 'P03_increment_by_index',
  'inputs_checked': 5,
  'max_abs_score_error': 0.0,
  'residual_max_abs_error': 0.0,
  'residual_mismatches': 0,
  'status': 'PASS'},
 {'program': 'P04_first_element',
  'inputs_checked': 5,
  'max_abs_score_error': 2.1022565244166458e-08,
  'residual_max_abs_error': 2.1022565244166458e-08,
  'residual_mismatches': 0,
  'status': 'PASS'},
 {'program': 'P05_histogram',
  'inputs_checked': 5,
  'max_abs_score_error': 1.9073486328125e-06,
  'residual_max_abs_error': 1.9073486328125e-06,
  'residual_mismatches': 0,
  'status': 'PASS'},
 {'program': 'P06_count_greater_than',
  'inputs_checked': 5,
  'max_abs_score_error': 1.9

In [51]:
trace_log_path = project_root / "results" / "logs" / "step2_manual_trace.json"
trace_log_path.parent.mkdir(parents=True, exist_ok=True)
trace_log_path.write_text(json.dumps(TRACE_LOG_ROWS, indent=2) + "\n")

print(f"Saved detailed block traces to {trace_log_path}")

Saved detailed block traces to /Users/raghul/RaspFormer/notebooks/results/logs/step2_manual_trace.json
